In [1]:
import cv2
import numpy as np
import transforms3d as tfs
import math
import open3d as o3d
import copy
import json

rgb_json = 'intrinsics_parameters.json' # Input the path to the intrinsic_rgb json file
robot_file = 'calibpos.txt'

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
rotation_vec_t2c, translation_t2c = get_rtvec_t2c(rgb_json)
print("Rotation_vec_t2c", rotation_vec_t2c)
print("TRanslation:", translation_t2c)
position_robot, quat_robot = get_robot_pos_quat(robot_file)

Rotation_vec_t2c [[-2.9925912595775928, 0.05740237952954883, -0.00957214886987871], [2.8798204306090818, -0.7886771381808992, -0.11754975532707752], [-2.8757800513886327, -0.802544136694838, -0.05811073804163084], [2.1308359670212655, -0.12274817584545694, 0.139568828670764], [2.0473485823808275, -0.5760419259520357, -0.34933644266442376], [2.295302794422795, 0.4956100158665835, 0.6191426146896908], [2.3787472957167592, -0.09715902332028052, 0.2979715265813223], [2.1879481715154867, -0.558728379021429, -0.27329459494186176], [2.32016073203568, 0.4979631612613104, 0.44761783143119643], [2.319475948758445, -0.10150365803139239, 0.2144734417576551], [2.336099082196418, -0.6857788043271003, -0.21681236756279165], [2.435908529373301, 0.5464341574313845, 0.3396138351639657], [2.9015388901181405, -0.05467643857412032, 1.1718697971205363], [2.7669125915328903, -0.9886529956982267, 1.0410704782366895], [2.6734100088240385, 0.9401829689594744, 1.3219418745084779], [2.8791119997412538, -0.0788638

In [ ]:
chess_to_cam = []
for i in range(len(rotation_vec_t2c)):
    test_var =translation_t2c[i] + rotation_vec_t2c[i]
    chess_to_cam.append(test_var)
print(chess_to_cam)
    
    

[[-105.17835389229226, 142.3949961468039, 831.5705753198948, -2.9925912595775928, 0.05740237952954883, -0.00957214886987871], [36.97407969623312, 165.6800170979073, 711.07152901909, 2.8798204306090818, -0.7886771381808992, -0.11754975532707752], [-270.7624077044642, -24.66389911964552, 768.6476852696255, -2.8757800513886327, -0.802544136694838, -0.05811073804163084], [-122.22692627076547, 124.76687479488719, 614.9298256266303, 2.1308359670212655, -0.12274817584545694, 0.139568828670764], [-46.71559389306271, 102.06069624393594, 539.6783375574253, 2.0473485823808275, -0.5760419259520357, -0.34933644266442376], [-157.00910912228215, 18.843250520051946, 488.04437148681484, 2.295302794422795, 0.4956100158665835, 0.6191426146896908], [-68.47003446432413, 93.93999581224662, 501.1297735713103, 2.3787472957167592, -0.09715902332028052, 0.2979715265813223], [-20.372329654196367, 70.15398942607021, 477.13901124028837, 2.1879481715154867, -0.558728379021429, -0.27329459494186176], [-178.147365550

In [3]:
def get_rtvec_t2c(intrinsincs_file): #intrinsincs_file is where intrinsic values from camera are stored
    try: 
        with open(intrinsincs_file, 'r') as intrinsic_data:
            data = json.load(intrinsic_data)
            rvecs = [[val[0] for val in vec] for vec in data["rvecs"]]
            tvecs = [[val[0] * 10 for val in vec] for vec in data["tvecs"]] # We multiply with 10 to change the tvecs from cm to mm
        return rvecs, tvecs  # return vectors for use in cv.calibrateHandEye 
    except:
        print("error loading file:"+ intrinsincs_file)
    

def get_robot_pos_quat(robot_file):
    coords = []
    quaternion = []

    with open(robot_file, "r") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        numbers = [float(x) for x in line.strip()[1:-1].split(",")] # We remove the brackets from the txt file and split whenever there is a comma
        
        if i % 2 == 0:
            coords.append(numbers)
        else:
            quaternion.append(numbers)

    return coords, quaternion

def quaternion_to_euler(x, y, z, w):

    
        t0 = +2.0 * (w * x + y * z)
        t1 = +1.0 - 2.0 * (x * x + y * y)
        X = math.degrees(math.atan2(t0, t1))

        t2 = +2.0 * (w * y - z * x)
        t2 = +1.0 if t2 > +1.0 else t2
        t2 = -1.0 if t2 < -1.0 else t2
        Y = math.degrees(math.asin(t2))

        t3 = +2.0 * (w * z + x * y)
        t4 = +1.0 - 2.0 * (y * y + z * z)
        Z = math.degrees(math.atan2(t3, t4))

        return X, Y, Z

In [ ]:
chess_to_cam_R,chess_to_cam_T = [],[]
end_to_base_R,end_to_base_T = [],[]
base_to_end_R,base_to_end_T = [],[]
for chess_cam in chess_to_cam:
    cc=chess_cam[3:6]
    cc_R, j = cv2.Rodrigues((cc[0],cc[1],cc[2]))
    chess_to_cam_R.append(cc_R)
    chess_to_cam_T.append(np.array(chess_cam[0:3]).reshape(3,1))
for end_base in end_to_base:
    ed=end_base[3:6]
    ed_R, j2 = cv2.Rodrigues((ed[0],ed[1],ed[2]))
    #end_to_base_R.append(tfs.euler.euler2mat(math.radians(ed[0]),math.radians(ed[1]),math.radians(ed[1]),axes='sxyz'))  #注意欧拉角顺序
    end_to_base_R.append(ed_R)
    end_to_base_T.append(np.array(end_base[0:3]).reshape(3,1))

    Trans=-np.array(end_base[0:3]).reshape(3,1) #齐次旋转矩阵的逆矩阵，位移部分不是单纯方向相反
    Trans=(ed_R.transpose())@Trans
    base_to_end_R.append(ed_R.transpose())
    base_to_end_T.append(Trans)
 
#print(chess_to_cam_R)
print("chess_to_cam_T is:",chess_to_cam_T)
#print(end_to_base_R)
print("base_to_end_T is:",base_to_end_T)
#eye to hand 输入的是base_to_end_R,base_to_end_T 关键函数
cam_to_base_R,cam_to_base_T = cv2.calibrateHandEye(base_to_end_R,base_to_end_T,chess_to_cam_R,chess_to_cam_T,
                                                 method=cv2.CALIB_HAND_EYE_TSAI)    
print("CAM TO BASE R:",cam_to_base_R)
print("CAM to BASE T:", cam_to_base_T)
cam_to_base_RT = tfs.affines.compose(np.squeeze(cam_to_base_T), cam_to_base_R, [1, 1, 1])    #squeeze 删除维度
print("标定结果：\n",cam_to_base_RT)
mesh = o3d.geometry.TriangleMesh.create_coordinate_frame()
mesh_t = copy.deepcopy(mesh).transform(cam_to_base_RT)
# 可视化
o3d.visualization.draw_geometries([mesh, mesh_t])
# 结果验证，原则上来说，每次结果相差较小
for i in range(0,numData):
    RT_base_to_end=np.column_stack((base_to_end_R[i],base_to_end_T[i].reshape(3,1)))
    RT_base_to_end=np.row_stack((RT_base_to_end,np.array([0,0,0,1])))
    # print(RT_end_to_base)
    RT_chess_to_cam=np.column_stack((chess_to_cam_R[i],chess_to_cam_T[i].reshape(3,1)))
    RT_chess_to_cam=np.row_stack((RT_chess_to_cam,np.array([0,0,0,1])))
    # print(RT_chess_to_cam)
    RT_chess_to_end=RT_base_to_end@cam_to_base_RT@RT_chess_to_cam #棋盘格相对于机器人末端坐标系位姿，固定
    # RT_chess_to_base=np.linalg.inv(RT_chess_to_base)
    print('第',i,'次')
    print(RT_chess_to_end)
    print('')